In [20]:
from core.algebra import Implies, Top

# Implies(a, b) = Top if a <= b, else b
# "if a is no stronger than b, the implication holds without restriction"
pairs = [
    (0.0, 0.0),
    (0.0, 1.0),
    (1.0, 0.0),
    (1.0, 1.0),
    (0.3, 0.7),
    (0.7, 0.3),
    (0.5, 0.5),
]

print(f"{'a':>6}  {'b':>6}  {'Implies(a,b)':>12}")
print("-" * 30)
for a, b in pairs:
    result = Implies(a, b)
    result_str = "Top" if result >= Top - 1 else f"{result:.4f}"
    print(f"{a:>6.1f}  {b:>6.1f}  {result_str:>12}")

     a       b  Implies(a,b)
------------------------------
   0.0     0.0           Top
   0.0     1.0           Top
   1.0     0.0        0.0000
   1.0     1.0           Top
   0.3     0.7           Top
   0.7     0.3        0.3000
   0.5     0.5           Top


In [21]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
from core.tensor import Tensor
from core.algebra import Implies

T = Tensor()

A = np.array([
    [0.9, 0.2, 0.0],
    [0.0, 0.7, 0.5],
    [0.3, 0.0, 0.8],
])

B = np.array([
    [0.6, 0.0, 0.4],
    [0.0, 0.8, 0.3],
    [0.5, 0.1, 0.9],
])

# Step 1: forward Join
C = T.Join(A, B, temp=0.0)
print("A =")
print(A)
print()
print("B =")
print(B)
print()
print("C = Join(A, B) =")
print(np.round(C, 4))
print()

# Step 2: Residuate — find greatest B' such that Join(A, B') <= C
B_prime = T.Residuate(A, C, temp=0.0)
print("B' = Residuate(A, C) =")
print(np.round(B_prime, 4))
print()

# Step 3: B' should be >= B (it is the upper bound)
print("B' >= B everywhere (B' is upper bound of B):", np.all(B_prime >= B - 1e-6))
print()

# Step 4: Join(A, B') should match C
C_check = T.Join(A, B_prime, temp=0.0)
print("Join(A, B') =")
print(np.round(C_check, 4))
print()
print("Join(A, B') matches C:", np.allclose(C_check, C, atol=1e-4))

A =
[[0.9 0.2 0. ]
 [0.  0.7 0.5]
 [0.3 0.  0.8]]

B =
[[0.6 0.  0.4]
 [0.  0.8 0.3]
 [0.5 0.1 0.9]]

C = Join(A, B) =
[[0.6 0.2 0.4]
 [0.5 0.7 0.5]
 [0.5 0.1 0.8]]

B' = Residuate(A, C) =
[[6.e-01 1.e-01 4.e-01]
 [5.e-01 1.e+09 5.e-01]
 [5.e-01 1.e-01 1.e+09]]

B' >= B everywhere (B' is upper bound of B): True

Join(A, B') =
[[0.6 0.2 0.4]
 [0.5 0.7 0.5]
 [0.5 0.1 0.8]]

Join(A, B') matches C: True


In [22]:
import torch
from core.algebra import Top

A_t = torch.tensor(A)
C_t = torch.tensor(C)

def fmt(M, tol=1e-3):
    """Format matrix, replacing Top with ∞."""
    rows = []
    for row in M:
        rows.append([" Top " if abs(v - Top) < tol else f"{v:.4f}" for v in row])
    col_w = max(len(c) for r in rows for c in r)
    return "\n".join("  [" + "  ".join(c.rjust(col_w) for c in row) + "]" for row in rows)

# torch lstsq: find B' that minimizes ||A @ B' - C||
result = torch.linalg.lstsq(A_t, C_t)
B_prime_torch = result.solution.numpy()
C_check_torch = (A_t @ torch.tensor(B_prime_torch)).numpy()

print("=== B (original) ===")
print(fmt(B))
print()
print("=== B' from Residuate (greatest solution) ===")
print(fmt(B_prime))
print()
print("=== B' from lstsq (approximate solution) ===")
print(fmt(B_prime_torch))
print()
print("Residuate B' >= B:", np.all((B_prime >= B - 1e-6) | (B_prime > Top - 1)))
print("lstsq     B' >= B:", np.all(B_prime_torch >= B - 1e-6))
print()
print("A @ lstsq B' matches C:", np.allclose(C_check_torch, C, atol=1e-4))

=== B (original) ===
  [0.6000  0.0000  0.4000]
  [0.0000  0.8000  0.3000]
  [0.5000  0.1000  0.9000]

=== B' from Residuate (greatest solution) ===
  [0.6000  0.1000  0.4000]
  [0.5000    Top   0.5000]
  [0.5000  0.1000    Top ]

=== B' from lstsq (approximate solution) ===
  [0.5730  0.0187  0.4195]
  [0.4213  0.9157  0.1124]
  [0.4101  0.1180  0.8427]

Residuate B' >= B: True
lstsq     B' >= B: False

A @ lstsq B' matches C: True
